In [372]:
from lttb import lttb
import numpy as np
import pandas as pd
import pdb
import plotly.graph_objects as go
import dtat.plot as dtatplot
from time import time

class PlotOrchestrator:
    def __init__(self, data):
        self.data = data
        self.lttb_data = data
        self.last_xrange = None
        self.requery_in_progress = False
        self.debounce_time = time()
        self.first_query = True
        
    def lttb(self, n_points, data=None):
        if data is None:
            data = self.data
        
        results = []

        # Group by 'name'
        for name, df in data.groupby("name"):
            # Drop duplicates and sort once
            df = df.drop_duplicates(subset="scet", keep="first").sort_values("scet")

            x = df["scet"].astype("int64").to_numpy()
            y = df["value"].to_numpy()

            
            if len(x) <= n_points:
                return data
            
            # Downsample using LTTB
            x1, y1 = lttb.downsample(np.column_stack([x, y]), n_points).T

            # Build decimated DataFrame
            decimated_df = pd.DataFrame({
                "scet": pd.to_datetime(x1),
                "value": y1,
                "name": name,
                "unit": df["unit"].iloc[0]
            })

            results.append(decimated_df)

        # Concatenate all at once (much faster)
        return pd.concat(results, ignore_index=True)

    
    def plot(self, data=None):
        if data is None: data = self.data
        fig,c,m,t = dtatplot.make_stacked_graph( # make_stacked_graph is the primary plotting method
            data,  # this is the data that DTAT plots from
            y_vars = [
                [c] for c in data['name'].unique()
            ],  # this is the y variables to plot
            multi_axis=True
            )  # the x variable to plot by defaults to SCET
        return fig
    
    def interactive_lttb_plot(self, n_points=500):
        figw = go.FigureWidget(self.plot(self.lttb(n_points)))
        figw.layout.on_change(self.requery_on_zoom, 'xaxis.range')
        self.fig = figw
        
        return figw

        
    def requery_on_zoom(self, relayout_data, zoomed_times):
        x_start = pd.to_datetime(zoomed_times[0])
        x_end = pd.to_datetime(zoomed_times[1])
        
        zoomed_data = self.data[(self.data["scet"] >= x_start) & (self.data["scet"] <= x_end)]
        
        dont_replot = True
        for trace in self.fig.data:
            if x_start > min(trace.x) or x_end < max(trace.x):
                dont_replot = False

        if dont_replot: 
            decimated_zoomed = self.lttb(500, self.data)
        else:
            decimated_zoomed = self.lttb(500, zoomed_data)

        # Update figure traces safely
        for trace in self.fig.data:
            trace_name = trace.name
            df_trace = decimated_zoomed[decimated_zoomed['name'] == trace_name]
            trace.x = df_trace['scet']
            trace.y = df_trace['value']


In [373]:
n_points = 100000
scet = np.linspace(0, 200 * np.pi, n_points)
value = np.sin(scet)

data = pd.DataFrame({
    'scet':  [pd.to_datetime(x*1e11) for x in scet] + [pd.to_datetime(x*1e11) for x in scet],
    'value': [np.sin(x) for x in scet] + [np.cos(x) for x in scet],
    'name': ['sin']*n_points + ['cos']*n_points,
    'unit': ['NA']*2*n_points
})

po = PlotOrchestrator(data)

#po.interactive_lttb_plot(500)

In [376]:
display(po.interactive_lttb_plot(500))

FigureWidget({
    'data': [{'hovertemplate': ('X: scet: %{x|%Y-%jT%H:%M:%S.%L' ... '%jT%H:%M:%S.%L}<extra></extra>'),
              'marker': {'color': '#0078AF',
                         'colorscale': [[0.0, 'rgb(0,0,131)'], [0.2,
                                        'rgb(0,60,170)'], [0.4, 'rgb(5,255,255)'],
                                        [0.6, 'rgb(255,255,0)'], [0.8,
                                        'rgb(250,0,0)'], [1.0, 'rgb(128,0,0)']],
                         'line': {'color': '#0073FF', 'width': 0.5},
                         'showscale': False,
                         'size': 5,
                         'symbol': 'circle'},
              'meta': [[None, Timestamp('1970-01-01 00:00:00')], [None,
                       Timestamp('1970-01-01 00:01:15.398977675')], [None,
                       Timestamp('1970-01-01 00:03:58.135104493')], ..., [None,
                       Timestamp('1970-01-01 17:23:23.142839512')], [None,
                       Timestamp('

In [375]:
print(po.a)
print(po.b)
print(po.time)
print(po.zt)

AttributeError: 'PlotOrchestrator' object has no attribute 'a'

In [371]:
print(po.a)
print(po.b)
print(po.time)
print(po.zt)

1970-01-01 02:45:47.037609064
1970-01-01T00:00:00.000000000
1766435518.649599
(9947037609064.223, 17403298433952.43)
